# Processamento de Vídeo utilizando o OpenCV

Este notebook mostra como implementar o processamento em tempo real de quadros de uma câmera utilizando boas práticas de programação e checagem de erros.

In [ ]:
import time
import cv2

class VideoStream:
    """Classe para gerenciar um stream de vídeo utilizando OpenCV. Inclui boas práticas de
    checagem de erros e gerenciamento de recursos."""
    def __init__(self, source=0, camera_settings=None, window_name="Video Stream"):
        """Inicializa um stream de vídeo.
        
        Args:
            source (int ou str): ID, URL ou caminho da fonte de vídeo.
            camera_settings (dict): Dicionário de configurações da câmera.
            window_name (str): Nome da janela onde o vídeo será exibido
        """

        self.source = source
        self.window_name = window_name
        # No Windows, pode ser necessário passar um segundo argumento cv2.CAP_DSHOW para evitar 
        # problemas com a câmera
        self.vcap = cv2.VideoCapture(self.source)
        if not self.vcap.isOpened():
            raise RuntimeError(f"Erro: Não foi possível acessar a câmera {self.source}")
        
        # Configurações da câmera (opcional)
        if camera_settings:
            for setting, value in camera_settings.items():
                self.vcap.set(setting, value)

    def start(self, processor):
        """Inicia o loop de captura e exibição do vídeo."""

        # Cria uma janela para exibir o vídeo. cv2.WINDOW_AUTOSIZE ajusta o tamanho da janela
        cv2.namedWindow(self.window_name, cv2.WINDOW_AUTOSIZE)
        # Dá uma pequena pausa para garantir que a câmera esteja pronta antes de começar a 
        # capturar frames
        time.sleep(0.5)

        try:
            while True:
                # Obtém próximo frame do vídeo. grabbed indica se o frame foi lido com sucesso, 
                # curr_frame possui a imagem
                grabbed, frame = self.vcap.read()
                if not grabbed:
                    print("Aviso: Não foi possível capturar o frame. Encerrando o stream.")
                    break
                    
                processed_frame = processor.process_frame(frame)
                
                # Exibe a imagem em uma janela nativa do SO
                cv2.imshow(self.window_name, processed_frame)
                
                # cv2.waitKey(x) faz o programa esperar x milisegundos para que uma tecla 
                # seja digitada. Dependendo da plataforma, o resultado dessa função pode 
                # ser um número com mais de 8 bits. O bitwise AND com 0xFF captura apenas
                # os 8 bits menos significativos.
                key = cv2.waitKey(1) & 0xFF
                
                # Se a tecla 'q' for pressionada, encerra o loop
                if key == ord("q"):
                    break
                # Se nenhuma tecla for pressionada, key possuirá o valor 255
                elif key != 255:
                    processor.on_key_press(key)
                    
        except KeyboardInterrupt:
            print("Stream interrompido pelo usuário.")
        except Exception as e:
            print(f"Ocorreu um erro: {e}")
        finally:
            # Garante que os recursos sejam liberados mesmo se ocorrer um erro ou interrupção
            self.stop()
    
    def stop(self):
        """Libera os recursos utilizados pelo stream de vídeo."""

        # Libera o dispositivo de captura
        self.vcap.release()
        # Fecha todas as janelas do OpenCV
        cv2.destroyAllWindows()
        # Comando extra para garantir que a janela feche corretamente no MacOS
        for _ in range(5):
            cv2.waitKey(1)

class VideoProcessor:
    """Classe de exemplo para processar os frames do vídeo. Neste caso, apenas retorna o 
    frame original."""
    def process_frame(self, frame):        
        return frame
    
    def on_key_press(self, key):
        """Exemplo de como lidar com eventos de teclado. Neste caso, apenas imprime a tecla 
        pressionada."""
        
        print(f"Tecla pressionada: {key}")
                

# ID da câmera. Em geral esse valor é 0 para a câmera integrada ou a primeira câmera USB conectada. 
# Se tiver mais de uma câmera, pode ser necessário testar com 1, 2, etc.
cam_id = 0

stream = VideoStream(cam_id)
processor = VideoProcessor()
stream.start(processor)

## Exemplo de processamento de vídeo

In [2]:
import numpy as np

class VideoProcessor:
    """Classe de exemplo para processar os frames do vídeo. Neste caso, aplica uma correção de
    gama e um limiar binário opcional. As teclas 'w' e 's' aumentam e diminuem o valor de gama, 
    respectivamente, enquanto a tecla 'l' ativa ou desativa a limiarização.
    """

    def __init__(self):
        self.gamma = 1.0
        self.apply_threshold = False
        
    def process_frame(self, frame):

        # Cria uma tabela de mapeamento para correção de gama
        table = (np.arange(256) / 255) ** self.gamma
        table = (255 * table).astype("uint8")
        # Aplica a correção de gama usando a função LUT do OpenCV
        processed_frame = cv2.LUT(frame, table)

        if self.apply_threshold:
            gray_frame = cv2.cvtColor(processed_frame, cv2.COLOR_BGR2GRAY)
            # Aplica um limiar binário simples. O valor 127 é o limiar, e 255 é o valor atribuído 
            # aos pixels que excedem o limiar
            _, processed_frame = cv2.threshold(gray_frame, 127, 255, cv2.THRESH_BINARY)

        return processed_frame

    def on_key_press(self, key):
        if key == ord('w'):
            self.gamma += 0.2
        elif key == ord('s'):
            # max evita que gamma fique negativo ou zero
            self.gamma = max(0.1, self.gamma - 0.2)
        elif key == ord('l'):
            self.apply_threshold = not self.apply_threshold


processor = VideoProcessor()
stream = VideoStream(cam_id)
stream.start(processor)

## Configurações da câmera

O OpenCV permite a alteração de parâmetros da câmera. Os possíveis parâmetros e respectivos valores dependem completamente do Sistema Operacional e do driver da câmera. 

Em sistemas Linux, é possível instalar a ferramenta v4l-utils para verificar informações da câmera. No Ubuntu/Mint/Debian basta digitar no terminal:

```bash
sudo apt install v4l-utils
```

In [3]:
# Lista os dispositivos de vídeo disponíveis no sistema. Útil para identificar o ID correto da câmera.
# Para cada dispositivo, o índice a ser usado no OpenCV é o valor X mostrado na primeira linha 
# contendo /dev/videoX
!v4l2-ctl --list-devices

FaceTime HD Camera (Built-in):  (usb-0000:00:1a.0-1.1):
	/dev/video0
	/dev/video1
	/dev/media0



In [4]:
# Lista os controles disponíveis para a câmera especificada por cam_id
!v4l2-ctl -d /dev/video{cam_id} --list-ctrls


User Controls

                     brightness 0x00980900 (int)    : min=-127 max=127 step=1 default=0 value=0
                       contrast 0x00980901 (int)    : min=0 max=64 step=1 default=32 value=32
                     saturation 0x00980902 (int)    : min=0 max=128 step=1 default=64 value=64
                            hue 0x00980903 (int)    : min=-4500 max=4500 step=100 default=0 value=0
        white_balance_automatic 0x0098090c (bool)   : default=1 value=1
                          gamma 0x00980910 (int)    : min=100 max=220 step=120 default=220 value=220
           power_line_frequency 0x00980918 (menu)   : min=0 max=2 default=3 value=3
      white_balance_temperature 0x0098091a (int)    : min=2800 max=6500 step=1 default=6500 value=4740 flags=inactive
                      sharpness 0x0098091b (int)    : min=0 max=16 step=1 default=8 value=8
         backlight_compensation 0x0098091c (int)    : min=0 max=1 step=1 default=1 value=1

Camera Controls

                  auto_

In [5]:
# Lista os formatos de vídeo suportados pela câmera 
!v4l2-ctl -d /dev/video{cam_id} --list-formats-ext

ioctl: VIDIOC_ENUM_FMT
	Type: Video Capture

	[0]: 'YUYV' (YUYV 4:2:2)
		Size: Discrete 160x120
			Interval: Discrete 0.033s (29.970 fps)
			Interval: Discrete 0.040s (25.000 fps)
			Interval: Discrete 0.042s (24.000 fps)
			Interval: Discrete 0.067s (15.000 fps)
		Size: Discrete 176x144
			Interval: Discrete 0.033s (29.970 fps)
			Interval: Discrete 0.040s (25.000 fps)
			Interval: Discrete 0.042s (24.000 fps)
			Interval: Discrete 0.067s (15.000 fps)
		Size: Discrete 320x240
			Interval: Discrete 0.033s (29.970 fps)
			Interval: Discrete 0.040s (25.000 fps)
			Interval: Discrete 0.042s (24.000 fps)
			Interval: Discrete 0.067s (15.000 fps)
		Size: Discrete 352x288
			Interval: Discrete 0.033s (29.970 fps)
			Interval: Discrete 0.040s (25.000 fps)
			Interval: Discrete 0.042s (24.000 fps)
			Interval: Discrete 0.067s (15.000 fps)
		Size: Discrete 640x480
			Interval: Discrete 0.033s (29.970 fps)
			Interval: Discrete 0.040s (25.000 fps)
			Interval: Discrete 0.042s (24.000 fps)
			Int

In [ ]:

camera_settings = {
    # Por padrão, o OpenCV utiliza o formato YUYV, que transmite os dados de vídeo sem compressão,
    # o que representa uma grande quantidade de dados e pode causar lentidão. O formato MJPG é 
    # uma opção de compressão que pode melhorar o desempenho
    cv2.CAP_PROP_FOURCC: cv2.VideoWriter_fourcc(*"MJPG"),
    # Configura a resolução do vídeo
    cv2.CAP_PROP_FRAME_WIDTH: 1280,
    cv2.CAP_PROP_FRAME_HEIGHT: 720
}

stream = VideoStream(cam_id, camera_settings)